# Week 1 Business Challenge: Generate Banking Brochure with OpenAI and translate with Ollama

In [12]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('AZURE_OPENAI_API_KEY')
endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')

if api_key and endpoint:
    print("Azure OpenAI key and endpoint found")
else:
    print("Set AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT in your .env file")

# MODEL is the name of your gpt-4o deployment in Azure (often also "gpt-4o")
MODEL = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')
openai = OpenAI(
    api_key=api_key,
    base_url=endpoint
)

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if not chunk.choices:  # Azure sends content-filter chunks with no choices
            continue
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

brochure = stream_brochure("HSBC", "https://www.hsbc.com/")

Azure OpenAI key and endpoint found
Selecting relevant links for https://www.hsbc.com/ by calling gpt-4o
Found 17 relevant links


# Welcome to HSBC  

## About Us  
Founded in **1865**, HSBC is one of the world’s largest and most renowned banking and financial services organizations. With a presence in **64 countries and territories**, HSBC connects customers around the world through its expertise in financial services and global capabilities. Our mission is to **open up a world of opportunity**, empowering individuals, businesses, and institutions to thrive and achieve their ambitions.  

## Our Strategy and Values  
At HSBC, we operate with a strong sense of purpose and commitment to sustainable development. Our strategy is built on leveraging our global network to connect people and businesses, while fostering innovation, inclusivity, and responsibility.  

Key values that guide us include:  
- **Dependable**: Acting with integrity, being reliable, and taking responsibility.  
- **Open**: Respecting others' ideas and cultures, and fostering inclusivity.  
- **Connected**: Building strong, lasting relationships and working effectively across diverse markets.  

## Customers and Services  
Our services span across:  
- **Wealth & Personal Banking**: Supporting individuals with savings, advisory, and financial management solutions.  
- **Commercial Banking**: Assisting businesses—big, small, and middle-market enterprises—in managing their finances, trade, and growth.  
- **Global Banking & Markets**: Providing tailored financial solutions for institutional and corporate clients.  
- **Digital Innovation**: Focused on integrating **AI**, **quantum technologies**, **digital accessibility**, and cutting-edge digital solutions to enhance customer experiences and operational efficiency.  

## Commitment to Sustainability  
Sustainability is at the heart of HSBC's strategy.  
- **Net Zero Transition Plan**: Pioneering initiatives to help clients transition to a low-carbon economy and achieve global climate goals.  
- **Climate Solutions Partnership**: Collaborating with customers and partners to develop innovative solutions addressing climate change.  
- **Responsible Business**: Emphasizing Environmental, Social, and Governance (ESG) standards, tackling issues like financial crime, modern slavery, health, and safety.  

## Global Presence  
HSBC serves customers in Europe, Asia, the Americas, the Middle East, and Africa. With connections to some of the most dynamic economies worldwide, we help our customers seize opportunities in an interconnected global economy.  

## Careers at HSBC  
At HSBC, we're committed to being an **inclusive** workplace where diversity thrives. We provide developmental opportunities for employees at all stages of their career. Our culture promotes innovation, collaboration, and a shared goal of making a positive impact in the world.  

Career highlights include:  
- **Professional Growth**: Opportunities for learning and development across global markets.  
- **Diversity & Inclusion**: A strong focus on creating equal opportunities and fostering an inclusive workplace.  
- **Innovative Environment**: Working on the cutting edge of **financial technology** and sustainability.  

## Join HSBC  
Whether you're a prospective customer, investor, or someone looking to grow your career, *HSBC is your partner for achieving ambitions and building a better future together*.  

Explore more about our services, vision, and career opportunities at [HSBC.com](https://www.hsbc.com)  

---  
**HSBC — Opening up a world of opportunity.**  

In [ ]:
#Translating in other languages with Ollama's TranslateGemma model
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434').rstrip('/')
if not OLLAMA_BASE_URL.endswith('/v1'):
    OLLAMA_BASE_URL += '/v1'  # Ollama's OpenAI-compatible API lives under /v1
OPEN_MODEL = "translategemma:4b"  # must match the tag shown by `ollama list`

llama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama")

# TranslateGemma expects this exact prompt as a single user message (no system prompt)
def get_translation_prompt(text, target_language="French", target_code="fr"):
    return f"""You are a professional English (en) to {target_language} ({target_code}) translator. Your goal is to accurately convey the meaning and nuances of the original English text while adhering to {target_language} grammar, vocabulary, and cultural sensitivities.
Produce only the {target_language} translation, without any additional explanations or commentary. Please translate the following English text into {target_language}:


{text}"""

def stream_translation(text, target_language="French", target_code="fr"):
    stream = llama.chat.completions.create(
        model=OPEN_MODEL,
        messages=[{"role": "user", "content": get_translation_prompt(text, target_language, target_code)}],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if not chunk.choices:
            continue
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response
stream_translation(brochure, "French", "fr")

# Bienvenue chez HSBC

## À propos de nous
Fondée en **1865**, HSBC est l'une des plus grandes et des plus réputées organisations bancaires et de services financiers au monde. Présente dans **64 pays et territoires**, HSBC met en relation les clients du monde entier grâce à son expertise en matière de services financiers et à ses capacités mondiales. Notre mission est de **ouvrir un monde d'opportunités**, en permettant aux individus, aux entreprises et aux institutions de prospérer et de réaliser leurs ambitions.

## Notre stratégie et nos valeurs
Chez HSBC, nous agissons avec un fort sens du but et un engagement en faveur du développement durable. Notre stratégie repose sur l'utilisation de notre réseau mondial pour connecter les personnes et les entreprises, tout en favorisant l'innovation, l'inclusion et la responsabilité.

Les valeurs clés qui nous guident comprennent :
- **Fiable**: Agir avec intégrité, être fiable et assumer ses responsabilités.
- **Ouvert**: Respecter les idées et les cultures des autres, et favoriser l'inclusion.
- **Connecté**: Établir des relations solides et durables, et travailler efficacement sur les différents marchés.

## Clients et services
Nos services couvrent les domaines suivants :
- **Banque privée et gestion de patrimoine**: Accompagner les particuliers grâce à des solutions de dépôt, de conseil et de gestion financière.
- **Banque commerciale**: Aider les entreprises – grandes, moyennes et petites –, à gérer leurs finances, leur commerce et leur croissance.
- **Banque mondiale et marchés**: Proposer des solutions financières sur mesure aux clients institutionnels et aux entreprises.
- **Innovation numérique**: Se concentrer sur l'intégration de l'**IA**, des **technologies quantiques**, de l'**accessibilité numérique** et des solutions numériques de pointe pour améliorer l'expérience client et l'efficacité opérationnelle.

## Engagement en faveur du développement durable
Le développement durable est au cœur de la stratégie de HSBC.
- **Plan de transition vers un objectif net zéro**: Mettre en œuvre des initiatives pour aider les clients à se tourner vers une économie à faible émission de carbone et à atteindre les objectifs climatiques mondiaux.
- **Partenariat pour des solutions climatiques**: Collaborer avec les clients et les partenaires pour développer des solutions innovantes répondant aux défis du changement climatique.
- **Entreprise responsable**: Mettre l'accent sur les normes ESG (environnementales, sociales et de gouvernance), et aborder des enjeux tels que la criminalité financière, la traite des êtres humains, la santé et la sécurité.

## Présence mondiale
HSBC dessert les clients en Europe, en Asie, en Amérique, au Moyen-Orient et en Afrique. Grâce à nos liens avec certaines des économies les plus dynamiques au monde, nous aidons nos clients à saisir les opportunités offertes par une économie mondiale interconnectée.

## Carrières chez HSBC
Chez HSBC, nous nous engageons à créer un environnement de travail **inclusif**, où la diversité est valorisée. Nous offrons des opportunités de développement aux employés à tous les stades de leur carrière. Notre culture encourage l'innovation, la collaboration et un objectif partagé : avoir un impact positif sur le monde.

Les points forts des carrières comprennent :
- **Développement professionnel**: Opportunités d'apprentissage et de développement sur les différents marchés.
- **Diversité et inclusion**: Un fort accent mis sur la création d'opportunités égales et sur la promotion d'un environnement de travail inclusif.
- **Environnement innovant**: Travailler à la pointe de la **technologie financière** et du développement durable.

## Rejoignez HSBC
Que vous soyez un client potentiel, un investisseur ou quelqu'un qui souhaite développer sa carrière, *HSBC est votre partenaire pour réaliser vos ambitions et construire un avenir meilleur ensemble*.

Explorez davantage nos services, notre vision et nos opportunités de carrière sur [HSBC.com](https://www.hsbc.com)

---
**HSBC – Ouvrir un monde d'opportunités.**

"# Bienvenue chez HSBC\n\n## À propos de nous\nFondée en **1865**, HSBC est l'une des plus grandes et des plus réputées organisations bancaires et de services financiers au monde. Présente dans **64 pays et territoires**, HSBC met en relation les clients du monde entier grâce à son expertise en matière de services financiers et à ses capacités mondiales. Notre mission est de **ouvrir un monde d'opportunités**, en permettant aux individus, aux entreprises et aux institutions de prospérer et de réaliser leurs ambitions.\n\n## Notre stratégie et nos valeurs\nChez HSBC, nous agissons avec un fort sens du but et un engagement en faveur du développement durable. Notre stratégie repose sur l'utilisation de notre réseau mondial pour connecter les personnes et les entreprises, tout en favorisant l'innovation, l'inclusion et la responsabilité.\n\nLes valeurs clés qui nous guident comprennent :\n- **Fiable**: Agir avec intégrité, être fiable et assumer ses responsabilités.\n- **Ouvert**: Respecte